In [1]:
import numpy as np
import time

# --- 1. Load Weights ---
print("Loading weights...")
data = np.load("cnn_weights.npz")
W_c1, b_c1 = data['c1_w'], data['c1_b']
W_c2, b_c2 = data['c2_w'], data['c2_b']
W_d1, b_d1 = data['d1_w'], data['d1_b']
W_d2, b_d2 = data['d2_w'], data['d2_b']

# --- 2. Define NumPy Layers ---

def relu(x):
    return np.maximum(0, x)

def conv1d_same(x, weights, bias):
    """
    Naïve 1D Convolution with 'same' padding.
    x: (Steps, In_Channels)
    weights: (Kernel_Size, In_Channels, Out_Filters)
    bias: (Out_Filters,)
    """
    steps, in_ch = x.shape
    k_size, _, out_filters = weights.shape
    
    # Calculate Padding for 'same' (TensorFlow style)
    pad_total = max(k_size - 1, 0)
    pad_beg = pad_total // 2
    pad_end = pad_total - pad_beg
    
    # Pad Input [Top, Bottom]
    x_padded = np.pad(x, ((pad_beg, pad_end), (0, 0)), mode='constant')
    
    output = np.zeros((steps, out_filters))
    
    # Sliding window operation
    for t in range(steps):
        # Extract window
        window = x_padded[t : t + k_size] # Shape (3, In_Channels)
        
        # Dot product: sum(Window * Weights) + Bias
        # This acts like a dense layer over the flattened window
        for f in range(out_filters):
            output[t, f] = np.sum(window * weights[:, :, f]) + bias[f]
            
    return output

def max_pooling_1d(x, pool_size=2):
    """
    x: (Steps, Channels)
    """
    steps, channels = x.shape
    new_steps = steps // pool_size
    output = np.zeros((new_steps, channels))
    
    for t in range(new_steps):
        # Take slice
        slice_ = x[t*pool_size : (t+1)*pool_size, :]
        # Max along time axis
        output[t, :] = np.max(slice_, axis=0)
        
    return output

def global_avg_pool(x):
    """Mean across the time steps"""
    return np.mean(x, axis=0)

# --- 3. The Model Function ---
def run_cnn_inference(window_input):
    # Input: (50, 3)
    
    # Layer 1: Conv1D (Relu)
    # Output: (50, 64)
    x = conv1d_same(window_input, W_c1, b_c1)
    x = relu(x)
    
    # Layer 2: MaxPool
    # Output: (25, 64)
    x = max_pooling_1d(x, pool_size=2)
    
    # Layer 3: Conv1D (Relu)
    # Output: (25, 32)
    x = conv1d_same(x, W_c2, b_c2)
    x = relu(x)
    
    # Layer 4: Global Avg Pool
    # Output: (32,) -> THIS IS YOUR ENTROPY SOURCE
    features = global_avg_pool(x)
    
    # Layer 5: Dense 1 (Relu)
    # Output: (32,)
    dense1 = np.dot(features, W_d1) + b_d1
    dense1 = relu(dense1)
    
    # Layer 6: Dense 2 (Linear)
    # Output: (3,)
    prediction = np.dot(dense1, W_d2) + b_d2
    
    return prediction, features

# --- 4. Helper Functions (ARX Mixer) ---
def rotate_left(val, r_bits, bit_width=32):
    return ((val << r_bits) & (2**bit_width - 1)) | (val >> (bit_width - r_bits))

def arx_mix(v0, v1):
    v0 = int(v0)
    v1 = int(v1)
    sum_val = (v0 + v1) & 0xFFFFFFFF
    rot_val = rotate_left(sum_val, 7)
    xor_val = rot_val ^ v0
    return xor_val

# --- 5. Main Execution Loop ---
print("Starting Generation on PYNQ (NumPy Accelerated)...")

N_BITS_NEEDED = 10000 # Keep small for Python speed check
generated_bits = []

# Random Seed Window (50 steps, 3 vars)
current_window = np.random.randn(50, 3)

start_time = time.time()

while len(generated_bits) * 32 < N_BITS_NEEDED:
    # A. Inference
    next_val, features = run_cnn_inference(current_window)
    
    # B. Update Window (Shift and Append)
    # Remove first row, add new prediction at end
    current_window = np.vstack([current_window[1:], next_val])
    
    # C. Post Processing
    # 1. Convert features to int32
    features_int = (features * 1e9).astype(np.int32) & 0xFFFFFFFF
    
    # 2. ARX Mix
    mixed_val = 0
    for i in range(0, 32, 2):
        mixed_val = arx_mix(mixed_val, features_int[i])
        
    # 3. STM Map
    x_val = next_val[0]
    # Normalize approx range -20 to 20 into 0.01 to 0.99
    stm_input = (x_val + 20) / 40.0
    stm_input = np.clip(stm_input, 0.01, 0.99)
    
    P_STM = 0.6
    if stm_input < P_STM:
        stm_out = stm_input / P_STM
    else:
        stm_out = (1 - stm_input) / (1 - P_STM)
        
    stm_int = int(stm_out * (2**32)) & 0xFFFFFFFF
    
    # 4. Final XOR
    final_32bit = mixed_val ^ stm_int
    generated_bits.append(final_32bit)
    
    if len(generated_bits) % 100 == 0:
        print(f"Generated {len(generated_bits)} blocks...")

end_time = time.time()
print(f"Finished. Time: {end_time - start_time:.2f}s")

# Save
final_array = np.array(generated_bits, dtype=np.uint32)
final_array.tofile("cnn_pynq_output.bin")
print("Saved to cnn_pynq_output.bin")

Loading weights...
Starting Generation on PYNQ (NumPy Accelerated)...
Generated 100 blocks...
Generated 200 blocks...
Generated 300 blocks...
Finished. Time: 205.05s
Saved to cnn_pynq_output.bin
